# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   


import sys
sys.path.append('../')
import helper_functions as hf

# generate recommendations

In [2]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 1
Very Important: Please Confirm the Iteration Number is Iteration 1
Very Important: Please Confirm the Iteration Number is Iteration 1


In [5]:
df_design, ax_client = hf.run_optimizer(current_iteration=n, n_trials=1)

[INFO 05-30 16:11:16] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 05-30 16:12:42] ax.service.ax_client: Generated new trial 6 with parameters {'s1': 100, 's2': 0, 's3': 80, 's4': 100, 's5': 9, 's6': 88, 's7': 100, 's8': 0, 's9': 28, 's10': 21, 's11': 38, 's12': 0, 'surfactant_conc': 1, 'drug_conc': 1} using model SAASBO.
[INFO 05-30 16:12:42] ax.service.ax_client: Saved JSON-serialized state of optimization to `optimizer/optimizer_1.json`.


# process results

In [3]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

[INFO 05-30 16:16:19] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'s1': 47, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 's9': 85, 's10': 76, 's11': 89, 's12': 88, 'surfactant_conc': 81, 'drug_conc': 86})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'s1': 93, 's2': 38, 's3': 50, 's4': 84, 's5': 50, 's6': 65, 's7': 78, 's8': 53, 's9': 26, 's10': 20, 's11': 36, 's12': 5, 'surfactant_conc': 29, 'drug_conc': 1})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'s1': 51, 's2': 77, 's3': 13, 's4': 24, 's5': 19, 's6': 94, 's7': 27, 's8': 92, 's9': 2, 's10': 28, 's11': 16, 's12': 30, 'surfactant_conc': 60, 'drug_conc': 53})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', parameters={'s1': 8, 's2': 20, 's3': 88, 's4': 72, 's5'

In [4]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [5]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: A1
Deep plate will start at: A5

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [6]:

hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_1.py


In [7]:
df_absorbance = hf.process_absorbance(iteration=n)

In [9]:
results = hf.build_results(n, df_conc, df_absorbance)

In [10]:
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,6,100,0,80,100,9,88,100,0,28,21,38,0,0.5,0.25,0,0.0,9


In [11]:
norm_results = hf.normalize_data(results, 'normalize')

In [12]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,6,100,0,80,100,9,88,100,0,28,21,38,0,0.01,0.25,0,0.0,0.75


# load the results to the optimizer

In [13]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 05-30 16:16:43] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 05-30 16:16:43] ax.service.ax_client: Completed trial 6 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.75, None)}.
[INFO 05-30 16:16:43] ax.service.ax_client: Saved JSON-serialized state of optimization to `optimizer/optimizer_1_loaded.json`.


AxClient(experiment=Experiment(drug_surfactant))